<a href="https://colab.research.google.com/github/HizoCoding/clayton-roy-personal-repository/blob/main/individual_encryption_pwd_mgr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Password Manager with XLSX Storage and Encryption

In [4]:
pip install openpyxl

**How to Use:**
**Note:** This functions best in colab because that is the editor I used to build the program and it has easy access to all of the listed packages. If this file is opened in a different code editor, you may need to pip install the fernet and openpyxl packages!

1. Run the entire program.
2. Set a master password that is used in perpetuity to access your local/cloud stored xlsx file that is created in the program.
3. Navigate with the interactive UI to add or edit any login information that you wish to store with encryption.


In [5]:
import importlib
import password_manager_lib
from cryptography.fernet import Fernet # Import Fernet for encryption.

# Reload the "password_manager_lib" module to ensure that changes made in the "%%writefile" cell are picked up.
importlib.reload(password_manager_lib)

# Import specific functions from the reloaded module.
from password_manager_lib import (
    load_key,
    verify_master_password,
    create_password_manager_ui
)

# Initialize 'fer' to None. It will be set after master password verification.
fer = None

# Call the master password verification function; proceeds to load the encryption key and display the UI if successful.
if verify_master_password():
    key = load_key()
    fer = Fernet(key)
    print("Master password verified. The password manager is ready.")
    create_password_manager_ui(fer)
else:
    # If master password verification fails, inform the user and explain limitations. UI will NOT be loaded.
    print("Master password verification failed. Application functionality will be limited.")
    print("The program will not be displayed as encryption key cannot be loaded.")

Enter your master password: Applicant1!2
Master password verified.
Master password verified. The password manager is ready.


Output()

This is the main entry point of the password manager application. It performs the following steps:

1.  **Imports necessary modules**: `importlib` for reloading the library, `password_manager_lib` for the core logic, and `Fernet` for encryption.
2.  **Reloads `password_manager_lib`**: Ensures that any changes made to the `password_manager_lib.py` file are applied without restarting the kernel.
3.  **Imports specific functions**: Brings `load_key`, `verify_master_password`, and `create_password_manager_ui` into the current scope.
4.  **Initializes `fer`**: This variable will hold the `Fernet` encryption instance.
5.  **Verifies Master Password**: Calls `verify_master_password()`.
    *   If successful, it loads the encryption key, creates a `Fernet` instance (`fer`), prints a success message, and then calls `create_password_manager_ui(fer)` to display the interactive user interface.
    *   If verification fails, it prints an error message and prevents the UI from loading, as the encryption key cannot be securely accessed.

In [6]:
%%writefile password_manager_lib.py
import cryptography
from cryptography.fernet import Fernet
import os
import hashlib
from tabulate import tabulate
import ipywidgets as widgets
from IPython.display import display, clear_output
import openpyxl

# Define file names for master password and stored passwords
MASTER_PWD_FILE = "master_pwd.key"
PASSWORDS_FILE = "passwords.xlsx" # Changed to .xlsx

def write_key():
    """Generates a new Fernet key and saves it to 'key.key'."""
    # Generate a new encryption key
    key = Fernet.generate_key()
    # Write the key to a file in binary mode
    with open("key.key", "wb") as key_file:
        key_file.write(key)

def load_key():
    # If the key file does not exist, create a new one
    if not os.path.exists("key.key"):
        write_key()
    # Open the key file in binary read mode
    file = open("key.key", "rb")
    key = file.read()
    file.close()
    return key

def hash_password(pwd):
    # Encode the password string to bytes and hash using SHA256
    return hashlib.sha256(pwd.encode()).hexdigest()

def set_master_password():
    #Prompts the user to set a new master password and saves its hash. Ensures the user confirms the password by entering it twice.
    while True:
        new_master_pwd = input("Set your new master password: ")
        confirm_master_pwd = input("Confirm your new master password: ")
        # Check if the entered passwords match
        if new_master_pwd == confirm_master_pwd:
            # Write the hashed master password to the designated file
            with open(MASTER_PWD_FILE, "w") as f:
                f.write(hash_password(new_master_pwd))
            print("Master password set successfully!")
            return True
        else:
            print("Passwords do not match. Please try again.")

def verify_master_password():
    # Check if the master password file exists
    if not os.path.exists(MASTER_PWD_FILE):
        print("No master password found. Please set one first.")
        # Attempt to set a new master password
        return set_master_password()

    # Read the stored master password hash
    with open(MASTER_PWD_FILE, "r") as f:
        stored_hash = f.read().strip()

    attempts = 0
    max_attempts = 3
    # Loop for a maximum number of attempts
    while attempts < max_attempts:
        entered_master_pwd = input("Enter your master password: ")
        # Compare the hash of the entered password with the stored hash
        if hash_password(entered_master_pwd) == stored_hash:
            print("Master password verified.")
            return True
        else:
            attempts += 1
            print(f"Incorrect master password. {max_attempts - attempts} attempts remaining.")
    print("Too many incorrect attempts. Exiting.")
    return False

def get_password_list(fernet_instance):
    password_data = []
    if not os.path.exists(PASSWORDS_FILE):
        # Create a new workbook with headers if the file doesn't exist
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Passwords"
        ws.append(["account", "password"])
        wb.save(PASSWORDS_FILE)
        return []

    wb = openpyxl.load_workbook(PASSWORDS_FILE)
    ws = wb.active

    # Skip header row
    for row in ws.iter_rows(min_row=2, values_only=True):
        if len(row) == 2 and row[0] is not None and row[1] is not None:
            try:
                user = str(row[0])
                passw_encrypted = str(row[1])
                passw_decrypted = fernet_instance.decrypt(passw_encrypted.encode()).decode()
                password_data.append({'account': user, 'password': passw_decrypted})
            except Exception as e:
                print(f"Error decrypting entry: {row} - {e}")
    return password_data

def save_password_list(fernet_instance, password_list):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Passwords"
    ws.append(["account", "password"])

    for entry in password_list:
        encrypted_pwd = fernet_instance.encrypt(entry['password'].encode()).decode()
        ws.append([entry['account'], encrypted_pwd])

    wb.save(PASSWORDS_FILE)

def add_new_password_entry(fernet_instance, account_name, password):
    current_passwords = get_password_list(fernet_instance)
    current_passwords.append({'account': account_name, 'password': password})
    save_password_list(fernet_instance, current_passwords)

def delete_password_entry(fernet_instance, account_name):
    current_passwords = get_password_list(fernet_instance)
    updated_passwords = [p for p in current_passwords if p['account'] != account_name]
    if len(updated_passwords) < len(current_passwords):
        save_password_list(fernet_instance, updated_passwords)
        return True
    return False

def update_password_entry(fernet_instance, old_account_name, new_account_name, new_password):
    current_passwords = get_password_list(fernet_instance)
    found = False
    for i, entry in enumerate(current_passwords):
        if entry['account'] == old_account_name:
            current_passwords[i]['account'] = new_account_name
            current_passwords[i]['password'] = new_password
            found = True
            break
    if found:
        save_password_list(fernet_instance, current_passwords)
    return found

def create_password_manager_ui(fernet_instance):
    output = widgets.Output()

    # Function to update and display the password list in the UI
    def update_password_display():
        with output:
            clear_output(wait=True)
            passwords = get_password_list(fernet_instance)
            if passwords:
                # Prepare data for tabulate, ensuring all values are strings
                table_data = [[str(entry['account']), str(entry['password'])] for entry in passwords]
                print(tabulate(table_data, headers="keys", tablefmt="grid"))
            else:
                print("No passwords stored.")

    # Initial display of passwords
    update_password_display()

    # UI elements for adding new password
    add_account_input = widgets.Text(description="Account:")
    add_password_input = widgets.Password(description="Password:")
    add_button = widgets.Button(description="Add Password")

    # UI elements for updating password
    old_account_input = widgets.Text(description="Old Account:")
    new_account_input = widgets.Text(description="New Account:")
    new_password_input = widgets.Password(description="New Password:")
    update_button = widgets.Button(description="Update Password")

    # UI elements for deleting password
    delete_account_input = widgets.Text(description="Account to Delete:")
    delete_button = widgets.Button(description="Delete Password")

    # Event handlers
    def on_add_button_clicked(b):
        add_new_password_entry(fernet_instance, add_account_input.value, add_password_input.value)
        add_account_input.value = ''
        add_password_input.value = ''
        update_password_display()
    add_button.on_click(on_add_button_clicked)

    def on_update_button_clicked(b):
        updated = update_password_entry(fernet_instance, old_account_input.value, new_account_input.value, new_password_input.value)
        if updated:
            old_account_input.value = ''
            new_account_input.value = ''
            new_password_input.value = ''
            update_password_display()
        else:
            with output:
                print(f"Account '{old_account_input.value}' not found for update.")
    update_button.on_click(on_update_button_clicked)

    def on_delete_button_clicked(b):
        deleted = delete_password_entry(fernet_instance, delete_account_input.value)
        if deleted:
            delete_account_input.value = ''
            update_password_display()
        else:
            with output:
                print(f"Account '{delete_account_input.value}' not found for deletion.")
    delete_button.on_click(on_delete_button_clicked)

    # Layout of the UI
    add_box = widgets.VBox([
        widgets.Label("Add New Password:"),
        add_account_input,
        add_password_input,
        add_button
    ])

    update_box = widgets.VBox([
        widgets.Label("Update Existing Password:"),
        old_account_input,
        new_account_input,
        new_password_input,
        update_button
    ])

    delete_box = widgets.VBox([
        widgets.Label("Delete Password:"),
        delete_account_input,
        delete_button
    ])

    display(widgets.HBox([add_box, update_box, delete_box]), output)

Overwriting password_manager_lib.py
